# Diagrama de Restricciones W/S – T/W para el UAV Transition

Este notebook utiliza los módulos de ADRpy para generar el diagrama de restricciones
de carga alar (W/S) vs relación empuje-peso (T/W) aplicado a un concepto aproximado
del UAV **Transition** (VTOL de ala fija).

**Autor:** Delpino  
**Tesis:** Análisis de diseño preliminar de aeronaves

## 1. Configuración del entorno

Configuramos el path del sistema para encontrar los módulos ADRpy en el directorio raíz del proyecto.

In [1]:
# Configuración de path para encontrar los módulos ADRpy
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Proyecto raíz agregado al path: {project_root}")

Proyecto raíz agregado al path: c:\Users\delpi\OneDrive\Tesis\ADRpy-VTOL


## 2. Importación de librerías

Importamos NumPy, Matplotlib y los módulos de ADRpy necesarios para el análisis de restricciones.

In [2]:
import os
# Limpiar variable de entorno problemática de matplotlib
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']

import numpy as np
import matplotlib.pyplot as plt
from ADRpy import constraintanalysis as ca
from ADRpy import atmospheres as at

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 3. Definición de diccionarios de parámetros

Definimos tres diccionarios con las claves exactas que utiliza `constraintanalysis.AircraftConcept`:

- **brief_transition**: Requerimientos de misión
- **design_transition**: Parámetros de diseño geométrico y masa
- **performance_transition**: Parámetros aerodinámicos y de rendimiento

In [3]:
# =============================================================================
# 1) Requerimientos de misión (brief): caso Transition
# =============================================================================
brief_transition = {
    # Aeródromo de referencia: asumimos nivel del mar
    "rwyelevation_m": 0.0,

    # Distancia de carrera de despegue. Transition es VTOL, pero para
    # reproducir el diagrama clásico W/S–T/W usamos un valor típico de
    # pequeño UAV ala fija (misma escala que el ejemplo de Keane).
    "groundrun_m": 60.0,

    # Sin viento ni pendiente en este ejemplo
    "to_headwind_kts": 0.0,
    "to_slope_perc": 0.0,

    # Ascenso inicial: tomamos nivel bajo
    "climbalt_m": 0.0,

    # Velocidades en nudos (kts); la base y la ficha dan m/s
    # V_stall FW (CAS) ≈ 14 m/s → ~27.2 kt
    # Usamos climb ≈ 1.3 * V_stall y turn ≈ crucero
    "climbspeed_kias": 35.0,             # ~1.3 * V_stall
    "secclimbspd_kias": 35.0,

    # Tasa de ascenso de la base (Tasa de ascenso ≈ 118 ft/min)
    "climbrate_fpm": 118.0,

    # Crucero: desde la base imputada
    # Altitud de crucero ~ 5000 m
    # Velocidad crucero IAS ≈ 20 m/s → ~38.9 kt
    "cruisealt_m": 5000.0,
    "cruisespeed_ktas": 39.0,

    # Suposición estándar: crucero con ~50% del thrust máximo
    "cruisethrustfact": 0.5,

    # Techo de servicio: ficha técnica 13 000 ft → ~3960 m
    "servceil_m": 3960.0,

    # Carga estructural de giro; usamos 2.5 g como valor típico
    "stloadfactor": 2.5,

    # Altitud de giro: vuelo relativamente bajo
    "turnalt_m": 500.0,

    # Giro a velocidad cercana a crucero
    "turnspeed_ktas": 39.0,

    # Velocidad de pérdida limpia CAS (14 m/s → ~27 kt)
    "vstallclean_kcas": 27.0,
}

print("Diccionario brief_transition definido.")
print(f"  - Velocidad de crucero: {brief_transition['cruisespeed_ktas']} KTAS")
print(f"  - Techo de servicio: {brief_transition['servceil_m']} m")

Diccionario brief_transition definido.
  - Velocidad de crucero: 39.0 KTAS
  - Techo de servicio: 3960.0 m


In [4]:
# =============================================================================
# 2) Parámetros de diseño geométrico y masa (design)
# =============================================================================
# NOTA: Todos los valores numéricos provienen de las tablas de la tesis (capítulo 8)
# o de supuestos explícitos documentados en el texto.

design_transition = {
    # Masa y peso
    "MTOW_kg": 18.0,                   # Masa máxima al despegue [kg]
    "weight_n": 176.6,                 # Peso en Newtons (≈18.0 × 9.81)
    
    # Geometría alar
    "wingarea_m2": 0.9289,             # Superficie alar [m²] - valor exacto de la tesis
    "aspectratio": 11.2,               # Relación de aspecto del ala
    "roottaperratio": 2.0,             # Relación cuerda raíz / cuerda punta
    "sweep_le_deg": 0.0,               # Flecha del borde de ataque [°]
    "sweep_mt_deg": 1.0,               # Flecha en línea de espesor medio [°]
    
    # Propulsión
    "thrust_n": 46.1,                  # Empuje estático del motor [N] - ficha técnica
    "bpr": -3,                         # Tipo propulsión: -3 = eléctrico sin correcciones

    # Fracciones de peso para cada fase de vuelo
    "weightfractions": {
        "turn": 1.0,
        "climb": 1.0,
        "cruise": 1.0,
        "servceil": 1.0,
    },
}

# Calcular punto de diseño
WS_design = design_transition["weight_n"] / design_transition["wingarea_m2"]
TW_design = design_transition["thrust_n"] / design_transition["weight_n"]

print("Diccionario design_transition definido (valores de la tesis).")
print(f"  - MTOW: {design_transition['weight_n']:.1f} N ({design_transition['MTOW_kg']:.1f} kg)")
print(f"  - Superficie alar: {design_transition['wingarea_m2']:.4f} m²")
print(f"  - Empuje estático: {design_transition['thrust_n']:.1f} N")

print(f"  - Relación de aspecto: {design_transition['aspectratio']}")
print(f"\n  → W/S diseño = {WS_design:.2f} N/m²")
print(f"  → T/W diseño = {TW_design:.4f}")

Diccionario design_transition definido (valores de la tesis).
  - MTOW: 176.6 N (18.0 kg)
  - Superficie alar: 0.9289 m²
  - Empuje estático: 46.1 N
  - Relación de aspecto: 11.2

  → W/S diseño = 190.12 N/m²
  → T/W diseño = 0.2610


In [5]:
# =============================================================================
# 3) Parámetros aerodinámicos y de performance (performance)
# =============================================================================
# NOTA: Valores ajustados según la tesis. CLmaxclean = 1.3 corresponde a un perfil
# NACA típico (2412 o similar) para UAV de bajo Reynolds. CD0 y k_induced basados
# en datos de diseño preliminar.

performance_transition = {
    # --- Densidades atmosféricas (ISA nivel del mar) ---
    "rho_cruise_kgm3": 1.225,
    "rho_TO_kgm3": 1.225,
    "rho_climb_kgm3": 1.225,
    "rho_turn_kgm3": 1.225,
    
    # --- Velocidades características [m/s] ---
    "V_cruise_ms": 27.8,               # ≈ 100 km/h - crucero
    "V_climb_ms": 16.7,                # ≈ 60 km/h - ascenso
    "V_app_ms": 13.0,                  # Aproximación
    "V_stall_clean_ms": 13.0,          # Pérdida en configuración limpia
    
    # --- Coeficientes aerodinámicos (valores de la tesis) ---
    # CLmaxclean = 1.3: valor final acordado para la tesis (perfil NACA típico)
    "CLmaxclean": 1.3,                 # CLmax limpio - VALOR DEFINITIVO TESIS
    "CLmax_TO": 1.7,                   # CLmax con flaps (despegue)
    "CLmax_L": 1.9,                    # CLmax con flaps (aterrizaje)
    "CLminclean": -0.8,                # CLmin limpio
    
    # Arrastre
    "CDminclean": 0.03,                # CD0 mínimo ala limpia (típico UAV)
    "CD0TO": 0.015,                    # CD0 en despegue
    "CDTO": 0.0898,                    # CD total en despegue
    "CL0TO": 0.2,                      # CL a alpha=0 en config. TO
    "CLTO": 0.97,                      # CL operativo en despegue
    "CLmaxTO": 1.7,                    # CLmax despegue (redundante, mantener)
    
    # Factor de arrastre inducido: k = 1/(π·AR·e) con AR=11.2 y e≈0.9
    "k_induced": 1.0 / (np.pi * 11.2 * 0.9),  # ≈ 0.0316
    
    # Pendiente de la curva CL–α
    "CLslope": 6.28,                   # ≈ 2π rad⁻¹
    
    # Coeficiente de rozamiento de pista
    "mu_R": 0.03,
    
    # --- Requisitos de misión ---
    "ROC_ms": 2.0,                     # Razón de ascenso [m/s]
    "loadfactor_turn": 2.0,            # Factor de carga en giro sostenido
    "sigma_servceil": 0.5,             # Densidad relativa en techo de servicio

    # Eficiencias propulsivas por fase
    "etaprop": {
        "take-off": 0.7,
        "climb":    0.7,
        "cruise":   0.7,
        "turn":     0.7,
        "servceil": 0.7,
    },
}

print("Diccionario performance_transition definido (valores de la tesis).")
print(f"  - CLmax limpio: {performance_transition['CLmaxclean']} (valor definitivo tesis)")
print(f"  - CDmin limpio: {performance_transition['CDminclean']}")
print(f"  - k_induced: {performance_transition['k_induced']:.4f}")
print(f"  - V crucero: {performance_transition['V_cruise_ms']} m/s")
print(f"  - ROC: {performance_transition['ROC_ms']} m/s")

Diccionario performance_transition definido (valores de la tesis).
  - CLmax limpio: 1.3 (valor definitivo tesis)
  - CDmin limpio: 0.03
  - k_induced: 0.0316
  - V crucero: 27.8 m/s
  - ROC: 2.0 m/s


## 4. Construcción del objeto AircraftConcept

Creamos el modelo atmosférico estándar y el concepto de aeronave utilizando los diccionarios definidos.

In [6]:
# Crear objeto atmósfera estándar (ISA)
atm = at.Atmosphere()

# Crear el concepto de aeronave con propulsión eléctrica
concept_transition = ca.AircraftConcept(
    brief_transition,
    design_transition,
    performance_transition,
    designatm=atm,
    propulsion="electric",
)

print("Objeto AircraftConcept creado exitosamente.")
print(f"Propulsión: eléctrica")

Objeto AircraftConcept creado exitosamente.
Propulsión: eléctrica


## 5. Cálculo de T/W requerido

Generamos un vector de carga alar W/S y calculamos las restricciones T/W para cada fase de vuelo.

In [7]:
# Vector de carga alar (Wing Loading) en N/m²
# Rango amplio para visualizar bien el diagrama de restricciones
ws_pa = np.linspace(20.0, 300.0, 400)  # N/m²

# Calcular las restricciones T/W para cada fase
twreq = concept_transition.twrequired(ws_pa)

print("Cálculo de T/W requerido completado.")
print(f"Rango de W/S analizado: {ws_pa[0]:.1f} - {ws_pa[-1]:.1f} N/m²")
print(f"\nRestricciones calculadas:")
for key in twreq.keys():
    if twreq[key] is not None:
        print(f"  - {key}")

Cálculo de T/W requerido completado.
Rango de W/S analizado: 20.0 - 300.0 N/m²

Restricciones calculadas:
  - take-off
  - liftoffspeed_mpstas
  - avspeed_mpstas
  - turn
  - turnfeasible
  - turncl
  - climb
  - cruise
  - servceil
  - combined


In [8]:
# Verificar las restricciones T/W en el punto de diseño
ws_transition = WS_design

# Encontrar el índice más cercano al W/S de diseño
idx_design = np.abs(ws_pa - ws_transition).argmin()

print(f"Punto de diseño Transition (valores de la tesis):")
print(f"  - W/S = {ws_transition:.2f} N/m²")
print(f"  - T/W = {TW_design:.4f}")

# Verificar estado de las restricciones individuales en el punto de diseño
print("\nRestricción T/W en el punto de diseño:")
tw_values = {}
for key in ["take-off", "climb", "cruise", "turn", "servceil"]:
    if twreq[key] is not None:
        tw_val = twreq[key][idx_design]
        tw_values[key] = tw_val
        print(f"  - {key}: {tw_val:.4f}")

# T/W mínimo requerido (máximo de todas las restricciones)
tw_min_required = max(v for v in tw_values.values() if not np.isnan(v))
print(f"\n✓ T/W mínimo requerido para el diseño: {tw_min_required:.4f}")

Punto de diseño Transition (valores de la tesis):
  - W/S = 190.12 N/m²
  - T/W = 0.2610

Restricción T/W en el punto de diseño:
  - take-off: 0.2487
  - climb: 0.1084
  - cruise: 0.1638
  - turn: 0.2714
  - servceil: 0.1034

✓ T/W mínimo requerido para el diseño: 0.2714


## 6. Diagrama de restricciones y análisis de márgenes

Generamos el diagrama de restricciones W/S–T/W estilo Keane, con la zona no factible coloreada y el análisis de márgenes del punto de diseño.

In [11]:
# =============================================================================
# Diagrama de restricciones W/S – T/W (estilo Keane)
# =============================================================================
import plotly.graph_objects as go

# --- Datos del punto de diseño ---
W_real = design_transition["weight_n"]
S_real = design_transition["wingarea_m2"]
T_real = design_transition["thrust_n"]
ws_real = WS_design
tw_real = TW_design

# --- Calcular la envolvente combinada ---
tw_stack = []
for key in ["take-off", "climb", "cruise", "turn", "servceil"]:
    if twreq[key] is not None:
        tw_stack.append(twreq[key])
tw_env = np.nanmax(np.vstack(tw_stack), axis=0)

# --- Etiquetas y colores (estilo imagen Keane) ---
etiquetas = {
    "turn": "Giro",
    "climb": "Ascenso", 
    "take-off": "Carrera T/O",
    "cruise": "Crucero",
    "servceil": "Techo serv."
}

# Colores apagados/crema con baja transparencia (relleno ARRIBA de cada curva = zona NO factible)
colores_fill = {
    "turn": "rgba(205, 175, 175, 0.28)",      # Rosa apagado/crema
    "climb": "rgba(175, 195, 205, 0.28)",      # Celeste apagado
    "take-off": "rgba(185, 165, 195, 0.28)",   # Lavanda suave
    "cruise": "rgba(215, 185, 185, 0.28)",     # Rosa crema
    "servceil": "rgba(170, 185, 205, 0.28)"    # Azul grisáceo
}

colores_linea = {
    "turn": "rgba(180, 130, 130, 0.7)",
    "climb": "rgba(100, 140, 160, 0.7)",
    "take-off": "rgba(140, 100, 160, 0.7)",
    "cruise": "rgba(190, 140, 150, 0.7)",
    "servceil": "rgba(110, 130, 170, 0.7)"
}

# Límite superior para el relleno
tw_max_fill = max(np.nanmax(tw_env), tw_real) * 1.3

# --- Crear figura ---
fig = go.Figure()

# 1) Áreas coloreadas para cada restricción (zona NO factible ARRIBA de cada curva)
# La zona factible es ARRIBA de la envolvente combinada
orden_plot = ["servceil", "cruise", "climb", "take-off", "turn"]

for key in orden_plot:
    if twreq[key] is not None:
        # Crear área desde la curva hasta el tope (zona NO factible está DEBAJO de cada restricción)
        fig.add_trace(go.Scatter(
            x=np.concatenate([ws_pa, ws_pa[::-1]]),
            y=np.concatenate([twreq[key], np.zeros_like(ws_pa)]),
            fill='toself',
            fillcolor=colores_fill[key],
            line=dict(width=0),
            name=etiquetas[key],
            hovertemplate=f"{etiquetas[key]}<br>W/S: %{{x:.1f}} N/m²<br>T/W: %{{y:.4f}}<extra></extra>",
            showlegend=True
        ))

# 2) Líneas de restricción (bordes suaves)
for key in orden_plot:
    if twreq[key] is not None:
        fig.add_trace(go.Scatter(
            x=ws_pa, 
            y=twreq[key],
            mode='lines',
            line=dict(color=colores_linea[key], width=1.2),
            showlegend=False,
            hovertemplate=f"{etiquetas[key]}<br>W/S: %{{x:.1f}} N/m²<br>T/W: %{{y:.4f}}<extra></extra>"
        ))

# --- Calcular márgenes ---
TW_min_req = np.interp(ws_real, ws_pa, tw_env)
idx_cross = np.where(tw_env <= tw_real)[0]
WS_min_req = ws_pa[idx_cross[0]] if len(idx_cross) > 0 else ws_pa[np.nanargmin(tw_env)]
delta_TW = tw_real - TW_min_req
delta_WS = ws_real - WS_min_req
pct_TW = (delta_TW / TW_min_req) * 100 if TW_min_req != 0 else 0

# Límite superior del gráfico
tw_max_plot = tw_max_fill

# 3) Punto de diseño (X negra)
fig.add_trace(go.Scatter(
    x=[ws_real], y=[tw_real],
    mode='markers',
    marker=dict(symbol='x-thin', size=16, line=dict(width=3, color='#444444'), color='white'),
    name=f"Diseño Transition<br>(W/S={ws_real:.1f}, T/W={tw_real:.3f})",
    hovertemplate=f"<b>Punto de diseño</b><br>W/S: {ws_real:.1f} N/m²<br>T/W: {tw_real:.4f}<extra></extra>"
))

# 4) Punto de T/W mínimo requerido
fig.add_trace(go.Scatter(
    x=[ws_real], y=[TW_min_req],
    mode='markers',
    name=f"T/W mín. requerido ({TW_min_req:.4f})",
    marker=dict(color='#8B4444', size=10, symbol='circle'),
    hovertemplate=f"<b>T/W mínimo requerido</b><br>W/S: {ws_real:.1f} N/m²<br>T/W: {TW_min_req:.4f}<extra></extra>"
))

# 5) Punto de W/S mínimo
fig.add_trace(go.Scatter(
    x=[WS_min_req], y=[tw_real],
    mode='markers',
    name=f"W/S mín. ({WS_min_req:.1f} N/m²)",
    marker=dict(color='#445588', size=10, symbol='circle-open', line=dict(width=2)),
    hovertemplate=f"<b>W/S mínimo</b><br>W/S: {WS_min_req:.1f} N/m²<br>T/W: {tw_real:.4f}<extra></extra>"
))

# 6) Líneas de proyección
fig.add_trace(go.Scatter(
    x=[ws_real, ws_real], y=[TW_min_req, tw_real],
    mode='lines', line=dict(color='#666666', dash='dot', width=1.2),
    showlegend=False, hoverinfo='skip'
))
fig.add_trace(go.Scatter(
    x=[WS_min_req, ws_real], y=[tw_real, tw_real],
    mode='lines', line=dict(color='#666666', dash='dot', width=1.2),
    showlegend=False, hoverinfo='skip'
))

# 7) Anotaciones (colores apagados)
fig.add_annotation(
    x=ws_real + 10, y=(tw_real + TW_min_req) / 2,
    text=f"<b>ΔT/W = {delta_TW:+.3f}</b><br>({pct_TW:+.1f}%)",
    showarrow=True, arrowhead=2, ax=35, ay=0,
    font=dict(size=11, color="#8B4444" if delta_TW < 0 else "#448844"),
    bgcolor="rgba(255,252,245,0.92)", bordercolor="#999999", borderwidth=1, borderpad=4
)
fig.add_annotation(
    x=(ws_real + WS_min_req) / 2, y=tw_real + 0.015,
    text=f"<b>ΔW/S = {delta_WS:+.1f} N/m²</b>",
    showarrow=False, font=dict(size=10, color="#445588"),
    bgcolor="rgba(255,252,245,0.92)", bordercolor="#999999", borderwidth=1, borderpad=3
)

# Anotación zona factible (estilo Keane) - centrado en zona blanca superior
fig.add_annotation(
    x=250, y=tw_max_plot * 0.65,
    text="<b>The feasible aeroplane lives<br>in this white space</b>",
    showarrow=False, font=dict(size=11, color="#555555"),
    bgcolor="rgba(255,252,245,0.0)", borderwidth=0,
    xanchor='center', yanchor='middle'
)

# 8) Layout
fig.update_layout(
    title=dict(
        text="<b>Diagrama de Restricciones W/S – T/W</b><br>" +
             "<sup>UAV Transition – caso de estudio</sup>",
        font=dict(size=18, color="#333333"), x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title="<b>W/S [N/m²]</b>", 
        range=[20, 320],
        showgrid=True,
        gridcolor='rgba(180,180,180,0.4)',
        gridwidth=1,
        zeroline=True,
        zerolinecolor='#888888',
        zerolinewidth=1,
        showline=True,
        linecolor='#666666',
        linewidth=1,
        mirror=True,
        tickfont=dict(size=11)
    ),
    yaxis=dict(
        title="<b>T/W [-]</b>", 
        range=[0, tw_max_plot],
        showgrid=True,
        gridcolor='rgba(180,180,180,0.4)',
        gridwidth=1,
        zeroline=True,
        zerolinecolor='#888888',
        zerolinewidth=1,
        showline=True,
        linecolor='#666666',
        linewidth=1,
        mirror=True,
        tickfont=dict(size=11)
    ),
    legend=dict(
        yanchor="top", y=0.99, 
        xanchor="right", x=0.99,
        font=dict(size=10), 
        bgcolor="rgba(255,252,245,0.92)",
        bordercolor="#999999", borderwidth=1
    ),
    plot_bgcolor='rgba(255,253,248,1)',  # Fondo ligeramente crema
    paper_bgcolor='white',
    width=900, height=650, 
    margin=dict(l=70, r=30, t=80, b=60),
    hovermode='closest'
)

fig.show()

# --- Resumen ---
print("\n" + "="*70)
print("RESUMEN DEL ANÁLISIS DE RESTRICCIONES")
print("="*70)
print(f"Aeronave: UAV Transition (VTOL ala fija)")
print(f"MTOW: {design_transition['MTOW_kg']:.1f} kg | S = {S_real:.4f} m² | T = {T_real:.1f} N")
print("-"*70)
print(f"Punto de diseño:  W/S = {ws_real:.2f} N/m²  |  T/W = {tw_real:.4f}")
print(f"Requerimientos:   T/W mín = {TW_min_req:.4f}  |  W/S mín = {WS_min_req:.1f} N/m²")
print("-"*70)
print(f"Márgenes: ΔT/W = {delta_TW:+.4f} ({pct_TW:+.1f}%)  |  ΔW/S = {delta_WS:+.1f} N/m²")
print("="*70)
if delta_TW > 0:
    print(f"\n✓ VIABLE: Exceso de empuje del {pct_TW:.1f}%")
else:
    T_necesario = W_real * TW_min_req
    print(f"\n⚠️ DÉFICIT: Falta {-pct_TW:.1f}% de empuje")
    print(f"   Opciones: aumentar T a {T_necesario:.1f} N o reducir W/S")


RESUMEN DEL ANÁLISIS DE RESTRICCIONES
Aeronave: UAV Transition (VTOL ala fija)
MTOW: 18.0 kg | S = 0.9289 m² | T = 46.1 N
----------------------------------------------------------------------
Punto de diseño:  W/S = 190.12 N/m²  |  T/W = 0.2610
Requerimientos:   T/W mín = 0.2717  |  W/S mín = 37.5 N/m²
----------------------------------------------------------------------
Márgenes: ΔT/W = -0.0107 (-3.9%)  |  ΔW/S = +152.6 N/m²

⚠️ DÉFICIT: Falta 3.9% de empuje
   Opciones: aumentar T a 48.0 N o reducir W/S
